In [2]:
import pandas as pd
import numpy as np

# 1. Cargar el dataset
df = pd.read_csv('loan_prediction_dataset.csv')

# 2. Vista previa
print("--- DATOS CARGADOS CON ÉXITO ---")
print(df.head())

# 3. Limpieza de valores nulos con las columnas correctas de tu CSV
df['Income'] = df['Income'].fillna(df['Income'].median())
df['Loan_Amount'] = df['Loan_Amount'].fillna(df['Loan_Amount'].median())
df['Credit_Score'] = df['Credit_Score'].fillna(df['Credit_Score'].median())

# 4. Crear la columna adaptada al entorno de arriendos (Relación Canon / Ingresos)
# Loan_Amount representa el Canon y Income representa los Ingresos
df['Relacion_Canon_Ingreso'] = df['Loan_Amount'] / (df['Income'] + 1)

print("\n--- ¡PROCESAMIENTO Y LIMPIEZA COMPLETADOS SIN ERRORES! ---")

--- DATOS CARGADOS CON ÉXITO ---
   Age  Income  Credit_Score  Loan_Amount  Loan_Term Employment_Status  \
0   56   81788           334        15022         48          Employed   
1   69  102879           781        21013         24     Self-Employed   
2   46   58827           779        39687         60     Self-Employed   
3   32  127188           364        16886         24        Unemployed   
4   60   25655           307        26256         36        Unemployed   

   Loan_Approved  
0              0  
1              1  
2              0  
3              0  
4              0  

--- ¡PROCESAMIENTO Y LIMPIEZA COMPLETADOS SIN ERRORES! ---


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE

# 1. Convertir texto a números (Employment_Status: Employed, Unemployed, etc.)
df_modelo = pd.get_dummies(df, columns=['Employment_Status'], drop_first=True)

# 2. Definir las variables de entrada (X) y lo que queremos predecir (y)
# Para nosotros, Loan_Approved = 0 significa mora/rechazado (1 en riesgo)
X = df_modelo.drop(columns=['Loan_Approved'])
y = df_modelo['Loan_Approved']

# 3. Aplicar SMOTE para balancear los datos sintéticamente
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# 4. Dividir en 80% Entrenamiento y 20% Pruebas
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# 5. Entrenar el modelo Random Forest
modelo_ml = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_ml.fit(X_train, y_train)

# 6. Evaluar con métricas (F1-Score y AUC-ROC)
y_pred = modelo_ml.predict(X_test)
y_prob = modelo_ml.predict_proba(X_test)[:, 1]

print("--- RESULTADOS DE EVALUACIÓN DEL MODELO ---")
print(classification_report(y_test, y_pred))
print(f"Área bajo la curva AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")

--- RESULTADOS DE EVALUACIÓN DEL MODELO ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       349
           1       1.00      1.00      1.00       315

    accuracy                           1.00       664
   macro avg       1.00      1.00      1.00       664
weighted avg       1.00      1.00      1.00       664

Área bajo la curva AUC-ROC: 1.0000


In [5]:
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# 1. Crear las variables de entrada y salida para la Lógica Difusa
prob_ml = ctrl.Antecedent(np.arange(0, 1.01, 0.01), 'prob_ml')
dias_retraso = ctrl.Antecedent(np.arange(0, 31, 1), 'dias_retraso')
indice_riesgo = ctrl.Consequent(np.arange(0, 101, 1), 'indice_riesgo')

# 2. Definir Funciones de Pertenencia (Bajo, Medio, Alto / Leve, Moderado, Critico)
prob_ml['baja'] = fuzz.trimf(prob_ml.universe, [0, 0, 0.5])
prob_ml['media'] = fuzz.trimf(prob_ml.universe, [0.2, 0.5, 0.8])
prob_ml['alta'] = fuzz.trimf(prob_ml.universe, [0.5, 1.0, 1.0])

dias_retraso['minimo'] = fuzz.trimf(dias_retraso.universe, [0, 0, 5])
dias_retraso['aceptable'] = fuzz.trimf(dias_retraso.universe, [3, 8, 15])
dias_retraso['critico'] = fuzz.trimf(dias_retraso.universe, [10, 30, 30])

indice_riesgo['bajo'] = fuzz.trimf(indice_riesgo.universe, [0, 0, 35])
indice_riesgo['moderado'] = fuzz.trimf(indice_riesgo.universe, [30, 50, 70])
indice_riesgo['alto'] = fuzz.trimf(indice_riesgo.universe, [65, 100, 100])

# 3. Base de Reglas IF-THEN (Definición del negocio)
regla1 = ctrl.Rule(prob_ml['baja'] & dias_retraso['minimo'], indice_riesgo['bajo'])
regla2 = ctrl.Rule(prob_ml['media'] & dias_retraso['aceptable'], indice_riesgo['moderado'])
regla3 = ctrl.Rule(prob_ml['alta'] | dias_retraso['critico'], indice_riesgo['alto'])

# 4. Sistema de Control y Simulación
sistema_control = ctrl.ControlSystem([regla1, regla2, regla3])
simulacion_fis = ctrl.ControlSystemSimulation(sistema_control)

# 5. Prueba de simulación con un caso real
# Supongamos: Probabilidad ML = 0.35 (35%) y Retraso histórico = 4 días
simulacion_fis.input['prob_ml'] = 0.35
simulacion_fis.input['dias_retraso'] = 4
simulacion_fis.compute()

score_final = simulacion_fis.output['indice_riesgo']

print("--- PRUEBA DEL MOTOR HÍBRIDO (ML + FIS) ---")
print(f"Probabilidad de mora previa (ML): {35}%")
print(f"Días de retraso histórico habituales: {4} días")
print(f"--> ÍNDICE DE RIESGO CALCULADO (0-100): {score_final:.2f}")

--- PRUEBA DEL MOTOR HÍBRIDO (ML + FIS) ---
Probabilidad de mora previa (ML): 35%
Días de retraso histórico habituales: 4 días
--> ÍNDICE DE RIESGO CALCULADO (0-100): 34.08


In [6]:
import joblib

# 1. Guardar el modelo de Machine Learning entrenado
joblib.dump(modelo_ml, 'modelo_riesgo_ml.pkl')

# 2. Guardar las columnas que usó el modelo para mantener el orden
joblib.dump(X.columns.tolist(), 'columnas_modelo.pkl')

print("--- MODELOS Y PIPELINE DE IA EXPORTADOS CORRECTAMENTE ---")
print("Archivos generados: 'modelo_riesgo_ml.pkl' y 'columnas_modelo.pkl'")

--- MODELOS Y PIPELINE DE IA EXPORTADOS CORRECTAMENTE ---
Archivos generados: 'modelo_riesgo_ml.pkl' y 'columnas_modelo.pkl'
